# Byte I/O - JavaScript

All 22 JavaScript examples from [docs/io.md](https://platob.github.io/yggdryl/io/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

In [ ]:
const assert = require('node:assert/strict')
const { IOBase } = require('yggdryl')

const handle = IOBase.fromBytes()
handle.pwrite(0, Buffer.from('symbol,price\n'))
handle.pwrite(13, Buffer.from('AAPL,1\n'))
assert.equal(handle.size, 20)

// Two reads at different offsets, in any order: there is no shared cursor.
assert.equal(handle.readRangeBytes(13, 4).toString(), 'AAPL')
assert.equal(handle.readRangeBytes(0, 6).toString(), 'symbol')

## Streamed bytes

In [ ]:
const assert = require('node:assert/strict')
const { IOBase } = require('yggdryl')

const handle = IOBase.fromBytes(Buffer.from('0123456789'))
assert.deepEqual(
  [...handle.pstreamBytes(2, 3)].map((part) => part.toString()),
  ['234', '567', '89'],
)

const cursor = handle.cursor(1)
const first = cursor.streamBytes(2).next()
assert.equal(first.value.toString(), '12')
assert.equal(cursor.tell(), 3)

## Laziness

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))

// Constructing touches nothing: no file is created, opened, or mapped.
const handle = new IOBase(path.join(root, 'nested', 'lazy.csv'))
assert.ok(!handle.exists())

// Reading something absent yields nothing rather than throwing.
assert.equal(handle.size, 0)
assert.equal(handle.readBytes().length, 0)

// Writing creates the resource, and any parent it needs.
handle.writeText('symbol,price\n')
assert.ok(handle.isFile())
assert.equal(handle.readText(), 'symbol,price\n')

fs.rmSync(root, { recursive: true, force: true })

## Kinds

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const folder = new IOBase(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-')))
assert.ok(folder.isDir())
assert.ok(!folder.isFile())

// Nothing is there, so nothing has decided; a write settles it.
const leaf = folder.joinpath('ticks.csv')
assert.ok(!leaf.exists())
leaf.writeText('symbol\n')
assert.ok(leaf.isFile())
assert.ok(!leaf.isDir())

fs.rmSync(folder.intoPath(), { recursive: true, force: true })

## Bytes or rows

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = new IOBase(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-')))

const notes = root.joinpath('notes.txt')
assert.ok(notes.isAtomic())
assert.ok(!notes.isTabular())

// The name is enough: nothing has been written to this location yet.
const trades = root.joinpath('trades.parquet')
assert.ok(trades.isTabular())
assert.ok(!trades.isAtomic())

assert.ok(!root.isAtomic())

fs.rmSync(root.intoPath(), { recursive: true, force: true })

## Whole values

In [ ]:
const assert = require('node:assert/strict')
const { IOBase } = require('yggdryl')

const handle = IOBase.fromBytes()
handle.writeBytes(Buffer.from('symbol,price\n'))

// `appendBytes` reports the offset the bytes landed at.
assert.equal(handle.appendBytes(Buffer.from('AAPL,1\n')), 13)
assert.equal(handle.readRangeBytes(0, 6).toString(), 'symbol')
// A range past the end yields what exists rather than throwing.
assert.equal(handle.readRangeBytes(100, 4).length, 0)
assert.equal(handle.readBytes().length, 20)

## Structured values

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase, Scalar } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-value-'))
const handle = new IOBase(path.join(root, 'trade.json.gz'))
handle.writeScalar({ quantity: 2, symbol: 'AAPL' })
const field = 'trade: struct<quantity: int32 not null, symbol: utf8 not null> not null'
assert.deepEqual(handle.readScalar(field), { quantity: 2, symbol: 'AAPL' })
const value = handle.readScalar({ field, scalar: true })
assert.ok(value instanceof Scalar)
assert.equal(value.kind, 'sequence')

## Cursors

In [ ]:
const assert = require('node:assert/strict')
const { IOBase } = require('yggdryl')

const handle = IOBase.fromBytes()
const cursor = handle.cursor()
cursor.write(Buffer.from('symbol,price\n'))

assert.equal(handle.readBytes().toString(), 'symbol,price\n')
cursor.seek(7)
assert.equal(cursor.read(5).toString(), 'price')

## What the bytes are

In [ ]:
const assert = require('node:assert/strict')
const { IOBase, MimeType } = require('yggdryl')

// Nothing names an in-memory buffer, so its type comes from its bytes.
const handle = IOBase.fromBytes(Buffer.from('{"symbol":"AAPL"}'))
assert.ok(handle.mediaType.base.equals(MimeType.JSON))

// It is re-derived after the content changes.
handle.writeBytes(Buffer.from('PAR1payload'))
assert.ok(handle.mediaType.base.equals(MimeType.PARQUET))

## Adding and removing a coding

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const plain = new IOBase(path.join(root, 'rows.json'))
plain.writeBytes(Buffer.from('{"symbol":"AAPL"}'))

// Nothing wraps these bytes, so there is nothing to undo.
assert.equal(plain.codec, null)

const encoded = new IOBase(path.join(root, 'rows.json.gz'))
assert.equal(encoded.codec, 'gzip')

// The target's name already said gzip, so nothing here repeats it.
assert.equal(plain.compressInto(encoded), encoded.size)
assert.deepEqual([...encoded.readBytes().subarray(0, 2)], [0x1f, 0x8b])

const decoded = new IOBase(path.join(root, 'roundtrip.json'))
assert.equal(encoded.decompressInto(decoded), 17)
assert.equal(decoded.readText(), '{"symbol":"AAPL"}')
assert.equal(decoded.codec, null)

// An in-memory target has no name to declare a coding, so this one is named.
const memory = IOBase.fromBytes()
assert.ok(plain.compressInto(memory, 'zstd') > 0)
assert.equal(memory.codec, 'zstd')

// A target declaring no coding is refused rather than copied unchanged.
assert.throws(
  () => plain.compressInto(new IOBase(path.join(root, 'copy.json'))),
  /expected a target declaring a content coding/,
)
assert.equal(fs.existsSync(path.join(root, 'copy.json')), false)

fs.rmSync(root, { recursive: true, force: true })

## Open and close

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const target = path.join(root, 'value.bin')
fs.writeFileSync(target, 'symbol,price\n')

const handle = new IOBase(target)
handle.open()
assert.equal(handle.opened(), true)
handle.close()
assert.equal(handle.closed(), true)

fs.rmSync(root, { recursive: true, force: true })

## Clearing and removing

In [ ]:
const assert = require('node:assert')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = path.join(os.tmpdir(), `yggdryl-docs-lifecycle-${process.pid}`)
const handle = new IOBase(path.join(root, 'logs'))
handle.mkdir()
handle.joinpath(['a.log']).writeText('line\n')

// Clearing empties the container and keeps it.
handle.clear()
assert.equal([...handle.ls(true, false)].length, 0)

// Removing deletes it; a second call succeeds, having done nothing.
handle.remove()
handle.remove()
assert.equal([...new IOBase(root).ls(false, false)].length, 0)

// The handle stays usable and lazy - a write recreates the resource.
const leaf = new IOBase(path.join(root, 'trades.csv'))
leaf.writeText('symbol,price\n')
leaf.remove()
assert.equal(leaf.exists(), false)
leaf.writeText('symbol,price\n')
assert.equal(leaf.readText(), 'symbol,price\n')
new IOBase(root).remove(true)

## Arrow batches

In [ ]:
const assert = require('node:assert/strict')
const arrow = require('apache-arrow')
const { BatchReader, Field, IOBase, MimeType, fields } = require('yggdryl')

// A non-null struct Field is the schema.
const schema = fields.struct(
  'row',
  [Field.from('id: int64'), Field.from('symbol: utf8')],
  { nullable: false },
)

const table = new arrow.Table({
  id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()),
  symbol: arrow.vectorFromArray(['AAPL', null], new arrow.Utf8()),
})

// The handle's own media type picks the encoding; no format argument is passed.
const handle = IOBase.fromBytes()
handle.mediaType = MimeType.ARROW_STREAM
const options = handle.recordOptions()

// The write path takes a batch reader and nothing else.
handle.overwriteArrowReader(BatchReader.from(table), options)
assert.ok(handle.readArrowField(options).equals(schema))

// The read path returns one. Batches arrive one at a time, never as a vector.
let rows = 0
for (const batch of handle.readArrowReader(options)) {
  rows += batch.numRows
}
assert.equal(rows, 2)

### Canonical record-write signatures

In [ ]:
const assert = require('node:assert/strict')
const { IOBase, MimeType } = require('yggdryl')

// An absent resource holds no batches rather than failing to parse.
const empty = IOBase.fromBytes()
empty.mediaType = MimeType.ARROW_STREAM
assert.equal([...empty.readArrowReader()].length, 0)

// An encoding this build does not implement is named rather than guessed.
const csv = IOBase.fromBytes()
csv.mediaType = MimeType.CSV
assert.throws(() => csv.recordOptions(), /text\/csv/)

## Rows as JavaScript objects

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const handle = new IOBase(path.join(root, 'trades.arrows'))
// Plain objects are rows; `overwriteRecords` widens them into the Arrow stream.
handle.overwriteRecords([
  { id: 1n, venue: 'XNAS' },
  { id: 2n, venue: null },
])

// Plain objects out, streamed batch by batch ...
assert.deepEqual([...handle.readRecords()].map((row) => row.id), [1n, 2n])

// ... or instances of any class whose constructor takes the plain row.
class Trade {
  constructor(row) {
    Object.assign(this, row)
  }
}
const trades = [...handle.readRecords(Trade)]
assert.ok(trades.every((t) => t instanceof Trade))

// An absent resource yields no records rather than raising.
assert.deepEqual([...new IOBase(path.join(root, 'absent.arrows')).readRecords()], [])

fs.rmSync(root, { recursive: true, force: true })

## Column pushdown

In [ ]:
const assert = require('node:assert/strict')
const arrow = require('apache-arrow')
const { BatchReader, Field, IOBase, MimeType, fields } = require('yggdryl')

const table = new arrow.Table({
  id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()),
  symbol: arrow.vectorFromArray(['AAPL', 'MSFT'], new arrow.Utf8()),
  venue: arrow.vectorFromArray(['XNAS', 'XNAS'], new arrow.Utf8()),
})

const handle = IOBase.fromBytes()
handle.mediaType = MimeType.ARROW_STREAM
handle.overwriteArrowReader(BatchReader.from(table))

// One of the three columns, declared as this read's schema.
const wanted = fields.struct('row', [Field.from('id: int64')], { nullable: false })
const options = handle.recordOptions()

const projected = handle.readArrowReader(options.withField(wanted))
assert.equal(projected.field.dtype.length, 1)
assert.equal(projected.intoTable().numCols, 1)

// The resource is unchanged: it still holds all three.
assert.equal(handle.readArrowField().dtype.length, 3)

// A column it does not hold cannot be projected out of it, so the encoding
// reads everything and the cast supplies that column as nulls.
const invented = fields.struct(
  'row',
  [Field.from('id: int64'), Field.from('nowhere: utf8?')],
  { nullable: false },
)
const widened = handle.readArrowReader(options.withField(invented))
assert.equal(widened.field.dtype.length, 2)

## Limiting a read or a write

In [ ]:
const assert = require('node:assert/strict')
const arrow = require('apache-arrow')
const { BatchReader, IOBase, MimeType } = require('yggdryl')

const table = new arrow.Table({
  id: arrow.vectorFromArray(
    Array.from({ length: 1000 }, (_, index) => BigInt(index)),
    new arrow.Int64(),
  ),
})

const handle = IOBase.fromBytes()
handle.mediaType = MimeType.ARROW_STREAM
handle.overwriteArrowReader(BatchReader.from(table))
const options = handle.recordOptions()

// Ten result rows, exactly: the batch the bound lands inside is sliced.
assert.equal(handle.readArrowReader(options.withMaxRowSize(10)).intoTable().numRows, 10)

// Zero is a valid ask: the shaped schema answers, and no batch flows.
const empty = handle.readArrowReader(options.withMaxRowSize(0))
assert.equal(empty.field.dtype.length, 1)
assert.equal(empty.intoTable().numRows, 0)

// A non-zero byte bound always yields at least one row.
assert.equal(handle.readArrowReader(options.withMaxByteSize(1)).intoTable().numRows, 1)

// A limited write truncates the data the caller offered: three rows land,
// and what the bound cut off is never pulled from the reader.
const copy = IOBase.fromBytes()
copy.mediaType = MimeType.ARROW_STREAM
copy.overwriteArrowReader(handle.readArrowReader(), options.withMaxRowSize(3))
assert.equal(copy.readArrowReader().intoTable().numRows, 3)

## Appending and merging

In [ ]:
const assert = require('node:assert/strict')
const arrow = require('apache-arrow')
const { BatchReader, Field, IOBase, MimeType, fields } = require('yggdryl')

const schema = fields.struct(
  'row',
  [Field.from('id: int64'), Field.from('symbol: utf8?')],
  { nullable: false },
)
const rows = (ids, symbols) =>
  BatchReader.from(
    new arrow.Table({
      id: arrow.vectorFromArray(ids, new arrow.Int64()),
      symbol: arrow.vectorFromArray(symbols, new arrow.Utf8()),
    }),
  )

const handle = IOBase.fromBytes()
handle.mediaType = MimeType.ARROW_STREAM
const options = handle.recordOptions().withField(schema)

// No match key: the resource is replaced.
handle.overwriteArrowReader(rows([1n, 2n], ['AAPL', 'MSFT']), options)

// Appending reads what is there, chains the new batches after it, and rewrites.
handle.appendArrowReader(rows([3n], ['NVDA']), options)
assert.equal(handle.readArrowReader(options).intoTable().numRows, 3)

// A match key merges: `2` is already stored and updates, `9` is new and appends.
const merging = options.withMergeByNames(['id'])
handle.mergeArrowReader(rows([2n, 9n], ['MSFT.O', 'AMD']), merging)
assert.equal(handle.readArrowReader(options).intoTable().numRows, 4)

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const handle = new IOBase(path.join(root, 'orders.arrows'))
handle.overwriteArrowTable(
  new arrow.Table({
    id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()),
    symbol: arrow.vectorFromArray(['AAPL', 'MSFT'], new arrow.Utf8()),
  }),
)

const narrowed = handle.recordOptions().withSelectByNames(['symbol'])
const table = handle.readArrowReader(narrowed).intoTable()
assert.deepEqual(table.schema.fields.map((field) => field.name), ['symbol'])

fs.rmSync(root, { recursive: true, force: true })

## Globbing and Hive partitions

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = path.join(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-')), 'lake')
for (const year of ['2024', '2025']) {
  const leaf = path.join(root, `year=${year}`, 'month=01')
  fs.mkdirSync(leaf, { recursive: true })
  fs.writeFileSync(path.join(leaf, 'part-0.parquet'), 'parquet')
}

const lake = new IOBase(root)

// A fixed prefix is descended, not listed and filtered.
assert.equal([...lake.glob('year=2024/**/*.parquet')].length, 1)
assert.equal([...lake.rglob('*.parquet')].length, 2)

// Partition filters select the leaves to overwrite or upsert.
const selected = [...lake.childrenWhere({ year: '2024' })]
assert.equal(selected.length, 1)
assert.deepEqual(selected[0].partitions, [
  { column: 'year', value: '2024' },
  { column: 'month', value: '01' },
])

fs.rmSync(root, { recursive: true, force: true })

## Partition pruning and filtering

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const lake = path.join(root, 'lake')
new IOBase(path.join(lake, 'year=2024', 'month=01', 'trades.arrows'))
  .overwriteRecords([{ id: 1n }, { id: 2n }])
new IOBase(path.join(lake, 'year=2024', 'month=02', 'trades.arrows'))
  .overwriteRecords([{ id: 3n }])

const handle = new IOBase(lake)
const options = handle.recordOptions().withFilterPartitions([
  ['year', '2024'],
  ['month', '01'],
])
assert.equal(handle.readArrowReader(options).intoTable().numRows, 2)

fs.rmSync(root, { recursive: true, force: true })

## Partition columns in the data

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { BatchReader, Field, IOBase, MimeType, RecordOptions, fields } = require('yggdryl')

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
fs.mkdirSync(path.join(root, 'year=2024', 'month=01'), { recursive: true })

const schema = fields.struct(
  'row',
  [Field.from('price: int64'), Field.from('year: int32'), Field.from('month: utf8')],
  { nullable: false },
)
const table = new arrow.Table({
  price: arrow.vectorFromArray([10n, 20n], new arrow.Int64()),
  year: arrow.vectorFromArray([2024, 2024], new arrow.Int32()),
  month: arrow.vectorFromArray(['01', '01'], new arrow.Utf8()),
})

// The rows carry every column; the write drops the two the path spells out.
const lake = new IOBase(root)
const options = RecordOptions.forMimeType(MimeType.ARROW_STREAM).withField(schema)
lake.overwriteArrowReader(BatchReader.from(table), options)

// Only `price` reached the leaf; the other two are the directory names.
const leaf = lake.joinpath('year=2024').joinpath('month=01').joinpath('part-0.arrows')
assert.equal(leaf.readArrowField().dtype.length, 1)

// Reading the folder restores them with their declared types.
const restored = lake.readArrowReader(options).intoTable()
assert.equal(restored.numCols, 3)
assert.equal(restored.schema.fields[1].type.toString(), 'Int32')
assert.deepEqual(restored.getChild('month').toArray(), ['01', '01'])

fs.rmSync(root, { recursive: true, force: true })